In [ ]:
# Enhanced SHM Live Analysis with Demo Mode (Generated Data)
import sys
import numpy as np
from PyQt5 import QtWidgets, QtCore, QtGui
import pyqtgraph as pg
from datetime import datetime
import csv


def moving_average(data, window_size=10):
    if len(data) < window_size:
        return np.array(data)
    return np.convolve(data, np.ones(window_size)/window_size, mode="valid")


class LivePlot(QtWidgets.QMainWindow):
    def __init__(self):
        super().__init__()
        
        self.is_running = False
        self.is_paused = False
        
        # Demo parameters for SHM simulation
        self.demo_time = 0.0
        self.demo_amplitude = 0.15  # 15 cm
        self.demo_omega = 2 * np.pi * 0.8  # frequency ~0.8 Hz
        self.demo_damping = 0.02  # small damping
        self.demo_phase = 0.0
        
        # Data storage
        self.time_values, self.x_values, self.v_values, self.a_values = [], [], [], []
        self.max_points = 300
        self.bar_N = 50
        self.v_bar_buffer = []
        self.a_bar_buffer = []
        self.last_values = {}
        
        # Setup UI
        self.init_ui()
        
    def init_ui(self):
        self.setWindowTitle("SHM Live Analysis - Demo Mode (Generated Data)")
        
        # Get screen geometry and adjust window size to avoid taskbar
        screen = QtWidgets.QApplication.primaryScreen().availableGeometry()
        window_width = min(1400, screen.width() - 100)
        window_height = min(850, screen.height() - 100)
        
        # Center the window on screen
        x = (screen.width() - window_width) // 2
        y = (screen.height() - window_height) // 2
        
        self.setGeometry(x, y, window_width, window_height)
        
        # Main widget and layout
        cw = QtWidgets.QWidget()
        self.setCentralWidget(cw)
        main_grid = QtWidgets.QGridLayout()
        cw.setLayout(main_grid)
        
        # Configure pyqtgraph
        pg.setConfigOptions(antialias=True, background="k", foreground="w")
        
        # Control panel at top
        self.create_control_panel(main_grid)
        
        # Plot grid
        self.plot_grid = QtWidgets.QGridLayout()
        main_grid.addLayout(self.plot_grid, 1, 0)
        
        # Create plots
        self.create_plots()
        
        # Sidebar
        self.create_sidebar(main_grid)
        
        # Timers
        self.timer = QtCore.QTimer()
        self.timer.timeout.connect(self.update)
        
        self.sidebar_timer = QtCore.QTimer()
        self.sidebar_timer.timeout.connect(self.update_sidebar)
        self.sidebar_timer.start(50)
        
    def create_control_panel(self, main_grid):
        """Create control panel with buttons"""
        control_widget = QtWidgets.QWidget()
        control_widget.setStyleSheet("background-color: #1a1a1a; padding: 10px;")
        control_layout = QtWidgets.QHBoxLayout()
        control_widget.setLayout(control_layout)
        
        # Demo controls
        demo_group = QtWidgets.QGroupBox("Demo Control")
        demo_group.setStyleSheet("color: white; font: 10pt;")
        demo_layout = QtWidgets.QHBoxLayout()
        
        self.start_btn = QtWidgets.QPushButton("▶ Start Demo")
        self.start_btn.setStyleSheet("background-color: #4CAF50; color: white; padding: 10px; font-weight: bold; font-size: 11pt;")
        self.start_btn.clicked.connect(self.toggle_demo)
        
        self.status_label = QtWidgets.QLabel("● Stopped")
        self.status_label.setStyleSheet("color: #ff4444; font-weight: bold; font-size: 11pt;")
        
        demo_layout.addWidget(self.start_btn)
        demo_layout.addWidget(self.status_label)
        demo_group.setLayout(demo_layout)
        
        # Data controls
        data_group = QtWidgets.QGroupBox("Data Control")
        data_group.setStyleSheet("color: white; font: 10pt;")
        data_layout = QtWidgets.QHBoxLayout()
        
        self.pause_btn = QtWidgets.QPushButton("⏸ Pause")
        self.pause_btn.setEnabled(False)
        self.pause_btn.setStyleSheet("background-color: #FF9800; color: white; padding: 8px; font-weight: bold;")
        self.pause_btn.clicked.connect(self.toggle_pause)
        
        self.clear_btn = QtWidgets.QPushButton("🗑 Clear Data")
        self.clear_btn.setStyleSheet("background-color: #f44336; color: white; padding: 8px; font-weight: bold;")
        self.clear_btn.clicked.connect(self.clear_data)
        
        self.export_btn = QtWidgets.QPushButton("💾 Export CSV")
        self.export_btn.setStyleSheet("background-color: #2196F3; color: white; padding: 8px; font-weight: bold;")
        self.export_btn.clicked.connect(self.export_data)
        
        self.screenshot_btn = QtWidgets.QPushButton("📷 Screenshot")
        self.screenshot_btn.setStyleSheet("background-color: #9C27B0; color: white; padding: 8px; font-weight: bold;")
        self.screenshot_btn.clicked.connect(self.take_screenshot)
        
        data_layout.addWidget(self.pause_btn)
        data_layout.addWidget(self.clear_btn)
        data_layout.addWidget(self.export_btn)
        data_layout.addWidget(self.screenshot_btn)
        data_group.setLayout(data_layout)
        
        # Settings
        settings_group = QtWidgets.QGroupBox("Settings")
        settings_group.setStyleSheet("color: white; font: 10pt;")
        settings_layout = QtWidgets.QVBoxLayout()
        
        # Points setting
        points_layout = QtWidgets.QHBoxLayout()
        self.points_spin = QtWidgets.QSpinBox()
        self.points_spin.setRange(100, 1000)
        self.points_spin.setValue(self.max_points)
        self.points_spin.setSingleStep(50)
        self.points_spin.setStyleSheet("background-color: white; padding: 5px;")
        self.points_spin.valueChanged.connect(self.update_max_points)
        points_layout.addWidget(QtWidgets.QLabel("Max Points:"))
        points_layout.addWidget(self.points_spin)
        
        # Mass setting
        mass_layout = QtWidgets.QHBoxLayout()
        self.mass_input = QtWidgets.QDoubleSpinBox()
        self.mass_input.setRange(0.1, 10.0)
        self.mass_input.setValue(0.750)
        self.mass_input.setSingleStep(0.05)
        self.mass_input.setDecimals(3)
        self.mass_input.setStyleSheet("background-color: white; padding: 5px;")
        mass_layout.addWidget(QtWidgets.QLabel("Mass (kg):"))
        mass_layout.addWidget(self.mass_input)
        
        # Demo parameters
        freq_layout = QtWidgets.QHBoxLayout()
        self.freq_input = QtWidgets.QDoubleSpinBox()
        self.freq_input.setRange(0.1, 5.0)
        self.freq_input.setValue(0.8)
        self.freq_input.setSingleStep(0.1)
        self.freq_input.setDecimals(2)
        self.freq_input.setStyleSheet("background-color: white; padding: 5px;")
        self.freq_input.valueChanged.connect(self.update_frequency)
        freq_layout.addWidget(QtWidgets.QLabel("Frequency (Hz):"))
        freq_layout.addWidget(self.freq_input)
        
        amp_layout = QtWidgets.QHBoxLayout()
        self.amp_input = QtWidgets.QDoubleSpinBox()
        self.amp_input.setRange(0.05, 0.5)
        self.amp_input.setValue(0.15)
        self.amp_input.setSingleStep(0.01)
        self.amp_input.setDecimals(2)
        self.amp_input.setStyleSheet("background-color: white; padding: 5px;")
        self.amp_input.valueChanged.connect(self.update_amplitude)
        amp_layout.addWidget(QtWidgets.QLabel("Amplitude (m):"))
        amp_layout.addWidget(self.amp_input)
        
        settings_layout.addLayout(points_layout)
        settings_layout.addLayout(mass_layout)
        settings_layout.addLayout(freq_layout)
        settings_layout.addLayout(amp_layout)
        settings_group.setLayout(settings_layout)
        
        # Add all groups to control layout
        control_layout.addWidget(demo_group)
        control_layout.addWidget(data_group)
        control_layout.addWidget(settings_group)
        control_layout.addStretch()
        
        main_grid.addWidget(control_widget, 0, 0, 1, 2)
        
    def create_plots(self):
        """Create all plot widgets"""
        # Plot 1: x & v
        self.plot1 = pg.PlotWidget(title="Displacement & Velocity vs Time")
        self.plot1.addLegend(offset=(20,20))
        self.plot1.setLabel('bottom', 'Time', units='s')
        self.plot1.setLabel('left', 'Displacement / Velocity', units='m or m/s')
        self.plot1.setMouseEnabled(x=False, y=False)  # Disable mouse interaction
        self.plot1.enableAutoRange(axis='xy', enable=True)  # Auto-scale based on data
        self.curve_x = self.plot1.plot(pen=pg.mkPen('y', width=2), name="x(t)")
        self.curve_v = self.plot1.plot(pen=pg.mkPen('c', width=2), name="v(t)")
        self.plot_grid.addWidget(self.plot1, 0, 0)
        
        # Plot 2: KE & PE
        self.plot2 = pg.PlotWidget(title="Energy Exchange")
        self.plot2.addLegend(offset=(20,20))
        self.plot2.setLabel('bottom', 'Time', units='s')
        self.plot2.setLabel('left', 'Energy', units='J')
        self.plot2.setMouseEnabled(x=False, y=False)  # Disable mouse interaction
        self.plot2.enableAutoRange(axis='xy', enable=True)  # Auto-scale based on data
        self.curve_ke = self.plot2.plot(pen=pg.mkPen('r', width=2), name="KE")
        self.curve_pe = self.plot2.plot(pen=pg.mkPen('g', width=2), name="PE")
        self.curve_te = self.plot2.plot(pen=pg.mkPen('y', width=2), name="TE")
        self.plot_grid.addWidget(self.plot2, 0, 1)
        
        # Plot 3: Moving bar |v| & |a|
        self.plot3 = pg.PlotWidget(title="|Velocity| & |Acceleration| over Time")
        self.plot3.addLegend(offset=(20,20))
        self.plot3.setLabel('bottom', 'Sample Index')
        self.plot3.setLabel('left', 'Magnitude')
        self.plot3.setMouseEnabled(x=False, y=False)  # Disable mouse interaction
        self.plot3.enableAutoRange(axis='xy', enable=True)  # Auto-scale based on data
        self.bar_v = pg.BarGraphItem(x=[], height=[], width=0.3, brush="c", name="|v|")
        self.bar_a = pg.BarGraphItem(x=[], height=[], width=0.3, brush="m", name="|a|")
        self.plot3.addItem(self.bar_v)
        self.plot3.addItem(self.bar_a)
        self.plot_grid.addWidget(self.plot3, 1, 0)
        
        # Plot 4: Phase portrait
        self.plot4 = pg.PlotWidget(title="Phase Portrait (x vs v)")
        self.plot4.setLabel('bottom', 'Displacement', units='m')
        self.plot4.setLabel('left', 'Velocity', units='m/s')
        self.plot4.setMouseEnabled(x=False, y=False)  # Disable mouse interaction
        self.plot4.enableAutoRange(axis='xy', enable=True)  # Auto-scale based on data
        self.curve_phase = self.plot4.plot(pen=pg.mkPen('w', width=2))
        self.plot_grid.addWidget(self.plot4, 1, 1)
        
    def create_sidebar(self, main_grid):
        """Create sidebar with live values"""
        self.sidebar_layout = QtWidgets.QVBoxLayout()
        main_grid.addLayout(self.sidebar_layout, 1, 1)
        
        self.sidebar_widget = QtWidgets.QWidget()
        self.sidebar_widget.setFixedWidth(350)
        self.sidebar_widget.setMinimumHeight(400)
        self.sidebar_widget.setMaximumHeight(700)
        self.sidebar_widget.setStyleSheet("background-color: black; border: 2px solid #333;")
        self.sidebar_layout.addWidget(self.sidebar_widget)
        
        self.sidebar_inner = QtWidgets.QVBoxLayout()
        self.sidebar_inner.setContentsMargins(15, 15, 15, 15)
        self.sidebar_inner.setSpacing(12)
        self.sidebar_widget.setLayout(self.sidebar_inner)
        
        # Title
        title = QtWidgets.QLabel("📊 Live Parameters")
        title.setStyleSheet("color: #4CAF50; font: bold 14pt; border-bottom: 2px solid #4CAF50; padding-bottom: 10px;")
        self.sidebar_inner.addWidget(title)
        
        # Demo mode indicator
        demo_label = QtWidgets.QLabel("🎮 DEMO MODE - Generated Data")
        demo_label.setStyleSheet("color: #FF9800; font: bold 10pt; background-color: #2a2a2a; padding: 8px; border-radius: 5px;")
        demo_label.setAlignment(QtCore.Qt.AlignCenter)
        self.sidebar_inner.addWidget(demo_label)
        
        self.labels = {}
        self.label_names = [
            "Displacement", "Velocity", "Acceleration",
            "Phase Angle", "Restoring Force",
            "Time Period", "Amplitude", "Frequency", "Angular Frequency",
            "KE", "PE", "TE", "Energy Dissipation Rate",
            "Max PE", "Max KE"
        ]
        
        for name in self.label_names:
            lbl = QtWidgets.QLabel(f"{name}: 0.00")
            lbl.setStyleSheet("color: white; font: 9pt; padding: 3px;")
            self.sidebar_inner.addWidget(lbl)
            self.labels[name] = lbl
        
        self.sidebar_inner.addStretch()
        
    def toggle_demo(self):
        """Start or stop the demo"""
        if not self.is_running:
            self.is_running = True
            self.start_btn.setText("⏹ Stop Demo")
            self.start_btn.setStyleSheet("background-color: #f44336; color: white; padding: 10px; font-weight: bold; font-size: 11pt;")
            self.status_label.setText("● Running")
            self.status_label.setStyleSheet("color: #4CAF50; font-weight: bold; font-size: 11pt;")
            self.pause_btn.setEnabled(True)
            self.timer.start(20)  # 50 Hz update rate
            QtWidgets.QMessageBox.information(self, "Demo Started", "Simulating SHM with generated data!")
        else:
            self.stop_demo()
            
    def stop_demo(self):
        """Stop the demo"""
        self.is_running = False
        self.timer.stop()
        self.start_btn.setText("▶ Start Demo")
        self.start_btn.setStyleSheet("background-color: #4CAF50; color: white; padding: 10px; font-weight: bold; font-size: 11pt;")
        self.status_label.setText("● Stopped")
        self.status_label.setStyleSheet("color: #ff4444; font-weight: bold; font-size: 11pt;")
        self.pause_btn.setEnabled(False)
        self.is_paused = False
        self.pause_btn.setText("⏸ Pause")
        self.pause_btn.setStyleSheet("background-color: #FF9800; color: white; padding: 8px; font-weight: bold;")
        
    def toggle_pause(self):
        """Pause or resume data collection"""
        self.is_paused = not self.is_paused
        if self.is_paused:
            self.pause_btn.setText("▶ Resume")
            self.pause_btn.setStyleSheet("background-color: #4CAF50; color: white; padding: 8px; font-weight: bold;")
        else:
            self.pause_btn.setText("⏸ Pause")
            self.pause_btn.setStyleSheet("background-color: #FF9800; color: white; padding: 8px; font-weight: bold;")
            
    def clear_data(self):
        """Clear all collected data"""
        reply = QtWidgets.QMessageBox.question(self, 'Clear Data', 
                                                'Are you sure you want to clear all data?',
                                                QtWidgets.QMessageBox.Yes | QtWidgets.QMessageBox.No)
        if reply == QtWidgets.QMessageBox.Yes:
            self.time_values.clear()
            self.x_values.clear()
            self.v_values.clear()
            self.a_values.clear()
            self.v_bar_buffer.clear()
            self.a_bar_buffer.clear()
            self.last_values.clear()
            self.demo_time = 0.0
            
    def export_data(self):
        """Export collected data to CSV"""
        if not self.time_values:
            QtWidgets.QMessageBox.warning(self, "No Data", "No data available to export!")
            return
            
        filename, _ = QtWidgets.QFileDialog.getSaveFileName(
            self, "Save Data", 
            f"shm_demo_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv",
            "CSV Files (*.csv)"
        )
        
        if filename:
            try:
                with open(filename, 'w', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerow(['Time (s)', 'Displacement (m)', 'Velocity (m/s)', 'Acceleration (m/s²)'])
                    for i in range(len(self.time_values)):
                        writer.writerow([
                            self.time_values[i],
                            self.x_values[i],
                            self.v_values[i],
                            self.a_values[i] if i < len(self.a_values) else 0
                        ])
                QtWidgets.QMessageBox.information(self, "Success", f"Data exported to:\n{filename}")
            except Exception as e:
                QtWidgets.QMessageBox.critical(self, "Export Error", f"Failed to export:\n{str(e)}")
                
    def take_screenshot(self):
        """Take screenshot of the application"""
        filename, _ = QtWidgets.QFileDialog.getSaveFileName(
            self, "Save Screenshot",
            f"shm_demo_screenshot_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png",
            "PNG Files (*.png)"
        )
        
        if filename:
            screen = QtWidgets.QApplication.primaryScreen()
            screenshot = screen.grabWindow(self.winId())
            screenshot.save(filename, 'png')
            QtWidgets.QMessageBox.information(self, "Success", f"Screenshot saved to:\n{filename}")
            
    def update_max_points(self, value):
        """Update maximum number of points to display"""
        self.max_points = value
        
    def update_frequency(self, value):
        """Update demo frequency"""
        self.demo_omega = 2 * np.pi * value
        
    def update_amplitude(self, value):
        """Update demo amplitude"""
        self.demo_amplitude = value
        
    def generate_shm_data(self):
        """Generate realistic SHM data with some noise"""
        # Add small random noise
        noise = np.random.normal(0, 0.002)
        
        # Damped SHM: x = A * e^(-γt) * cos(ωt + φ)
        damping_factor = np.exp(-self.demo_damping * self.demo_time)
        x = self.demo_amplitude * damping_factor * np.cos(self.demo_omega * self.demo_time + self.demo_phase) + noise
        
        # Velocity: v = dx/dt
        v = -self.demo_amplitude * damping_factor * self.demo_omega * np.sin(self.demo_omega * self.demo_time + self.demo_phase)
        v += -self.demo_damping * self.demo_amplitude * damping_factor * np.cos(self.demo_omega * self.demo_time + self.demo_phase)
        v += np.random.normal(0, 0.01)
        
        return self.demo_time, x, v
        
    def update(self):
        """Main update loop"""
        if not self.is_running or self.is_paused:
            return
            
        try:
            # Generate SHM data
            t, x, v = self.generate_shm_data()
            self.demo_time += 0.02  # 20ms increment
            
            # Compute acceleration
            if len(self.v_values) > 1:
                a = (v - self.v_values[-1]) / (t - self.time_values[-1] + 1e-6)
            else:
                a = 0.0
            
            # Append values
            self.time_values.append(t)
            self.x_values.append(x)
            self.v_values.append(v)
            self.a_values.append(a)
            
            # Keep latest points
            if len(self.time_values) > self.max_points:
                self.time_values = self.time_values[-self.max_points:]
                self.x_values = self.x_values[-self.max_points:]
                self.v_values = self.v_values[-self.max_points:]
                self.a_values = self.a_values[-self.max_points:]
            
            # Smooth data
            x_s = moving_average(self.x_values, 8)
            v_s = moving_average(self.v_values, 8)
            a_s = moving_average(self.a_values, 8)
            t_s = self.time_values[-len(x_s):]
            
            # Plot 1
            self.curve_x.setData(t_s, x_s)
            self.curve_v.setData(t_s[-len(v_s):], v_s)
            
            # Plot 2: Energy
            m = self.mass_input.value()
            if len(x_s) > 0:
                x_max = np.max(np.abs(x_s))
                v_max = np.max(np.abs(v_s))
                k = m * (v_max / (x_max + 1e-6))**2
            else:
                k = (2*np.pi)**2 * m
            
            KE = 0.5*m*v_s**2
            PE = 0.5*k*x_s**2
            TE = KE + PE
            KE_s = moving_average(KE, 8)
            TE_s = moving_average(TE, 8)
            
            self.curve_ke.setData(t_s[-len(KE_s):], KE_s)
            self.curve_pe.setData(t_s, PE)
            self.curve_te.setData(t_s[-len(TE_s):], TE_s)
            
            # Plot 3: bars
            self.v_bar_buffer.append(abs(v_s[-1]))
            self.a_bar_buffer.append(abs(a_s[-1]))
            if len(self.v_bar_buffer) > self.bar_N:
                self.v_bar_buffer = self.v_bar_buffer[-self.bar_N:]
                self.a_bar_buffer = self.a_bar_buffer[-self.bar_N:]
            indices = np.arange(len(self.v_bar_buffer))
            self.bar_v.setOpts(x=indices-0.15, height=self.v_bar_buffer, width=0.3)
            self.bar_a.setOpts(x=indices+0.15, height=self.a_bar_buffer, width=0.3)
            
            # Plot 4
            self.curve_phase.setData(x_s, v_s)
            
            # Store last values
            self.last_values = {
                "Displacement": x_s[-1],
                "Velocity": v_s[-1],
                "Acceleration": a_s[-1],
                "Phase Angle": np.arctan2(v_s[-1], x_s[-1]),
                "Restoring Force": -k*x_s[-1],
                "Time Period": 2*np.pi/np.sqrt(k/m),
                "Amplitude": np.max(np.abs(x_s)),
                "Frequency": np.sqrt(k/m)/(2*np.pi),
                "Angular Frequency": np.sqrt(k/m),
                "KE": KE[-1],
                "PE": PE[-1],
                "TE": TE[-1],
                "Energy Dissipation Rate": (TE[-1]-TE[-2])/(t_s[-1]-t_s[-2]+1e-6) if len(TE)>1 else 0.0,
                "Max PE": np.max(PE),
                "Max KE": np.max(KE)
            }
            
        except Exception as e:
            print("Update error:", e)
    
    def update_sidebar(self):
        """Update sidebar values"""
        for key, val in self.last_values.items():
            if key in self.labels:
                self.labels[key].setText(f"{key}: {val:.6f}")
                
    def closeEvent(self, event):
        """Handle window close event"""
        if self.is_running:
            self.stop_demo()
        event.accept()


if __name__ == "__main__":
    
    app = QtWidgets.QApplication(sys.argv)
    app.setStyle('Fusion')  # Modern look
    win = LivePlot()
    win.show()
    sys.exit(app.exec_())